In [1]:
import sys
sys.path.append("..")
from benchmarks.guacamol.assess_goal_directed_generation import assess_goal_directed_from_smiles
import pandas as pd


conditions = {
    "alzheimer": ["AChE", "MAOB"],
    "schizophrenia": ["D2R", "_5HT2A"],
    "parkinson": ["D2R", "D3R"],
}

smiles_dict = {}

# Load generated molecules SMILES
for disease, targets in conditions.items():
    targets_file = "_".join(targets) + "_SUM.csv"
    smiles_list = pd.read_csv(f"../generated_molecules/25-epoch/{targets_file}")["SMILES"].tolist()
    smiles_dict[disease] = smiles_list

compo_gpt_results = assess_goal_directed_from_smiles(
    smiles_dict, json_output_file=None, benchmark_version="multitarget"
)
results_df = pd.DataFrame(compo_gpt_results["results"])
results_df["Approach"] = "CoMPO-GPT"
results_df

Running benchmark 1/3: alzheimer
alzheimer: 92 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 92 instead of 100. Padding scores with 8 zeros...
  Score: 0.739042
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 88 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 88 instead of 100. Padding scores with 12 zeros...
  Score: 0.832637
  Execution time: 0:00:00
Running benchmark 3/3: parkinson
parkinson: 89 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 89 instead of 100. Padding scores with 11 zeros...
  Score: 0.814479
  Execution time: 0:00:00


,benchmark_name,score,optimized_molecules,execution_time,number_scoring_function_calls,metadata,Approach
0,alzheimer,0.739042,"[(COc1cccc(C2CC(=O)c3ccccc3O2)c1OC, 0.80484128...",0,92,"{'top_1': 0.804841283032323, 'top_10': 0.77692...",CoMPO-GPT
1,schizophrenia,0.832637,"[(O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)ccc2N1, ...",0,88,"{'top_1': 0.9063821446603729, 'top_10': 0.8760...",CoMPO-GPT
2,parkinson,0.814479,"[(CCCN1CCN(c2cccc(Cl)c2Cl)CC1, 0.8694780437434...",0,89,"{'top_1': 0.8694780437434, 'top_10': 0.8593910...",CoMPO-GPT


In [2]:
from pathlib import Path
import numpy as np

# Ensure runtime warnings raise as errors to debug
import warnings
warnings.filterwarnings(
    action='error', message='',
    category=RuntimeWarning
)


METHODS = {"CoMPO-GPT": compo_gpt_results}

# Baselines
baselines = [
    "DeepLig",
    "POLYGON",
    "MTMol-GPT",
]

diseases = ["alzheimer", "schizophrenia", "parkinson"]

for baseline in baselines:
    print(baseline)
    # List of diseases to evaluate
    diseases = ["alzheimer", "schizophrenia", "parkinson"]
    
    # Load generated molecules for each disease
    smiles_dict = {}
    
    for disease in diseases:
        path = f"../generated_molecules/{baseline}/{disease}-mpo.csv"
        if Path(path).exists():
            df = pd.read_csv(path,header=None)
            smiles_dict[disease] = df[0].tolist()
        else:
            print(f"Warning: No file found for {baseline} - {disease}")

    # Run benchmarks and store results
    METHODS[baseline] = assess_goal_directed_from_smiles(
        smiles_dict=smiles_dict,
        json_output_file=None, 
        benchmark_version="multitarget"
    )

METHODS.keys()

DeepLig
Running benchmark 1/3: alzheimer
alzheimer: 51 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 51 instead of 100. Padding scores with 49 zeros...
  Score: 0.658534
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
schizophrenia: 65 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 65 instead of 100. Padding scores with 35 zeros...
  Score: 0.749057
  Execution time: 0:00:00
Running benchmark 3/3: parkinson
parkinson: 36 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 36 instead of 100. Padding scores with 64 zeros...
  Score: 0.657533
  Execution time: 0:00:00
POLYGON
Running benchmark 1/3: alzheimer
alzheimer: 99 instead of 100
[WARNING] An incorrect number of distinct molecules was generated: 99 instead of 100. Padding scores with 1 zeros...
  Score: 0.634525
  Execution time: 0:00:00
Running benchmark 2/3: schizophrenia
  Score: 0.686692
  Execution time: 0:00:00


dict_keys(['CoMPO-GPT', 'DeepLig', 'POLYGON', 'MTMol-GPT'])

In [3]:
metric_order = [
    "Score",
    "Target Response",
    "Blood-Brain Barrier",
    "CNS MPO",
    "Synthetic Accessibility",
]

# benchmark_order = [
#     "Alzheimer MPO",
#     "Schizophrenia MPO",
#     "Parkinson MPO",
# ]
benchmark_order = [
    "alzheimer",
    "schizophrenia",
    "parkinson",
]

report_results = list()

def get_metadata_keys(metadata):
    keys = list(metadata.keys())

    target = [k for k in keys if "GeometricMeanScoringFunction" in k]
    target_scores = metadata[target[0]] if len(target) > 0 else None

    bbb = [k for k in keys if "BBBResponseScoringFunction" in k]
    bbb_scores = metadata[bbb[0]] if len(bbb) > 0 else None

    sa = [k for k in keys if "SyntheticAccessibilityScoringFunction" in k]
    sa_scores = metadata[sa[0]] if len(sa) > 0 else None

    cns = [k for k in keys if "CNS_MPO_ScoringFunction" in k]
    cns_scores = metadata[cns[0]] if len(cns) > 0 else None

    return {
        "Target Response": target_scores,
        "Blood-Brain Barrier": bbb_scores,
        "Synthetic Accessibility": sa_scores,
        "CNS MPO": cns_scores        
    }


for method, report in METHODS.items():
    results = report["results"]
        
    for result in results:
        results_info = dict()

        results_info["Method"] = method
        results_info["Benchmark"] = result["benchmark_name"]
        results_info["Score"] = f"{result['score']:.5f}"

        metadata = result["metadata"]
        scores_dict = get_metadata_keys(metadata)

        for score_key in scores_dict:
            scores = scores_dict[score_key]
            if scores is None:
                continue

            mean_score = np.mean(scores)
            std_score = np.std(scores)

            results_info[score_key] = f"{mean_score:.3f} ± {std_score:.3f}"
        
        report_results.append(results_info.copy())

# Reorder the benchmarks
df = pd.DataFrame(report_results)   
df["Benchmark"] = pd.Categorical(df["Benchmark"], benchmark_order, ordered=True)
dfs = list()

for metric in metric_order:
    df_score = df.pivot_table(
        index=["Benchmark"],
        columns=["Method"],
        values=[metric],
        aggfunc=lambda x: x,
    )

    df_score.columns = df_score.columns.droplevel(0)
    df_score["Metric"] = metric

    dfs.append(df_score)

concat_dfs = pd.concat(dfs, axis=0).sort_values(by=["Benchmark"]).reset_index()

concat_dfs["Metric"] = pd.Categorical(
    concat_dfs["Metric"], categories=metric_order, ordered=True
)

# [["Benchmark", 'Metric'] + list(METHODS.keys())]
concat_dfs.set_index(["Benchmark", "Metric"])

Method                                     CoMPO-GPT        DeepLig   
Benchmark     Metric                                                  
alzheimer     Score                          0.73904        0.65853  \
              Target Response          0.504 ± 0.058  0.366 ± 0.076   
              Blood-Brain Barrier      0.652 ± 0.194  0.881 ± 0.112   
              CNS MPO                  0.886 ± 0.142  0.900 ± 0.055   
              Synthetic Accessibility  0.834 ± 0.059  0.931 ± 0.053   
schizophrenia Score                          0.83264        0.74906   
              Target Response          0.718 ± 0.086  0.568 ± 0.067   
              Blood-Brain Barrier      0.903 ± 0.124  0.891 ± 0.106   
              CNS MPO                  0.854 ± 0.083  0.910 ± 0.047   
              Synthetic Accessibility  0.805 ± 0.064  0.939 ± 0.044   
parkinson     Score                          0.81448        0.65753   
              Target Response          0.720 ± 0.086  0.501 ± 0.263   
              Blood-Brain Barrier      0.872 ± 0.123  0.908 ± 0.110   
              CNS MPO                  0.833 ± 0.088  0.906 ± 0.049   
              Synthetic Accessibility  0.813 ± 0.063  0.941 ± 0.060   

Method                                     MTMol-GPT        POLYGON  
Benchmark     Metric                                                 
alzheimer     Score                          0.75416        0.63452  
              Target Response          0.494 ± 0.070  0.607 ± 0.023  
              Blood-Brain Barrier      0.614 ± 0.204  0.334 ± 0.124  
              CNS MPO                  0.940 ± 0.099  0.536 ± 0.085  
              Synthetic Accessibility  0.846 ± 0.087  0.755 ± 0.013  
schizophrenia Score                          0.80506        0.68669  
              Target Response          0.562 ± 0.189  0.593 ± 0.028  
              Blood-Brain Barrier      0.757 ± 0.226  0.871 ± 0.022  
              CNS MPO                  0.912 ± 0.100  0.664 ± 0.002  
              Synthetic Accessibility  0.817 ± 0.102  0.606 ± 0.016  
parkinson     Score                          0.80694        0.69481  
              Target Response          0.558 ± 0.133  0.623 ± 0.035  
              Blood-Brain Barrier      0.703 ± 0.250  0.905 ± 0.068  
              CNS MPO                  0.917 ± 0.101  0.721 ± 0.004  
              Synthetic Accessibility  0.844 ± 0.099  0.481 ± 0.054

In [4]:
print(
    concat_dfs
        .set_index(["Benchmark", "Metric"])
        .to_latex()
)

\begin{tabular}{llllll}
\toprule
 & Method & CoMPO-GPT & DeepLig & MTMol-GPT & POLYGON \\
Benchmark & Metric &  &  &  &  \\
\midrule
\multirow[t]{5}{*}{alzheimer} & Score & 0.73904 & 0.65853 & 0.75416 & 0.63452 \\
 & Target Response & 0.504 ± 0.058 & 0.366 ± 0.076 & 0.494 ± 0.070 & 0.607 ± 0.023 \\
 & Blood-Brain Barrier & 0.652 ± 0.194 & 0.881 ± 0.112 & 0.614 ± 0.204 & 0.334 ± 0.124 \\
 & CNS MPO & 0.886 ± 0.142 & 0.900 ± 0.055 & 0.940 ± 0.099 & 0.536 ± 0.085 \\
 & Synthetic Accessibility & 0.834 ± 0.059 & 0.931 ± 0.053 & 0.846 ± 0.087 & 0.755 ± 0.013 \\
\cline{1-6}
\multirow[t]{5}{*}{schizophrenia} & Score & 0.83264 & 0.74906 & 0.80506 & 0.68669 \\
 & Target Response & 0.718 ± 0.086 & 0.568 ± 0.067 & 0.562 ± 0.189 & 0.593 ± 0.028 \\
 & Blood-Brain Barrier & 0.903 ± 0.124 & 0.891 ± 0.106 & 0.757 ± 0.226 & 0.871 ± 0.022 \\
 & CNS MPO & 0.854 ± 0.083 & 0.910 ± 0.047 & 0.912 ± 0.100 & 0.664 ± 0.002 \\
 & Synthetic Accessibility & 0.805 ± 0.064 & 0.939 ± 0.044 & 0.817 ± 0.102 & 0.606 ± 0.